# 第 9 课 · XPath

> **内核提醒（每课都要看）**：本 notebook 必须在 conda 环境 **ai_learn** 中运行。
> 右上角内核应显示 `Python 3.10.21 ('ai_learn')`；打开后从上到下逐格运行（Shift+Enter）。
> 若报 `ModuleNotFoundError`，先检查内核是否选成了 ai_learn。

> BeautifulSoup 有个老搭档 **lxml**，它自带另一套定位语言 **XPath**。
> Scrapy、很多老项目默认就用 XPath，必须会读。本课只引入：`fromstring`、`//标签`、`text()`、`@属性`、谓语 `[...]`。

In [1]:
from pathlib import Path
from lxml import html as lxml_html     # lxml 的 HTML 解析子模块
print('lxml 导入成功')

DATA_DIR = Path(r'C:\Users\千禧\Doubao\chats\2026-09-23\new-chat\AI学习\08_爬虫\data')
books_html = (DATA_DIR / 'books_index.html').read_text(encoding='utf-8')
quotes_html = (DATA_DIR / 'quotes.html').read_text(encoding='utf-8')

lxml 导入成功


## 9.1 lxml.html.fromstring：得到另一棵树

**生活化类比**：BeautifulSoup 和 lxml 是两位管理员。一位递卡片（CSS 选择器），一位递**坐标尺**（XPath）。
书是同一本书，量法不同而已。

**是什么**：`tree = lxml_html.fromstring(html_text)` 得到一棵树对象；`tree.xpath('表达式')` 返回一个**列表**。

**为什么需要**：XPath 能表达「找第几个」「包含某文本」这类 CSS 写起来别扭的条件，Scrapy 默认用它。

In [2]:
tree = lxml_html.fromstring(books_html)
print('类型：', type(tree))
all_p = tree.xpath('//p')          # //p = 在整页任意位置找所有 <p>
print('一共有多少个 p：', len(all_p))

类型： <class 'lxml.html.HtmlElement'>
一共有多少个 p： 13


## 9.2 text() 与 @属性：直接取文字和属性

**是什么**：
- `//p/text()`：节点里的**纯文字**；
- `//a/@href`：节点的**属性值**（`@` 后面跟属性名）。

**为什么需要**：很多时候你要的不是节点本身，而是它肚子里的字或属性，XPath 能一句话直接取出来。

In [3]:
prices = tree.xpath('//p[@class="price_color"]/text()')   # 所有价格
print('全部价格：', prices)

titles = tree.xpath('//h3/a/@title')                        # 所有书名（a 的 title 属性）
print('全部书名：', titles)

全部价格： ['¥51.77', '¥53.74', '¥50.10', '¥47.82']
全部书名： ['A Light in the Attic', 'Tipping the Velvet', 'Soumission', 'Sharp Objects']


## 9.3 谓语 [ ]：第几个、按条件筛

**是什么**：
- `//article[1]`：**第一个** article。注意 **XPath 下标从 1 开始，不是 0**（和 Python 列表不一样！）；
- `//p[contains(@class,"price_color")]`：class 里**包含** price_color 的 p（对付一个标签有多个 class）。

**为什么需要**：真实网页一个标签常有多个 class，精确相等 `=` 会匹配不到，必须用 `contains`。

In [4]:
first_price = tree.xpath('//p[contains(@class,"price_color")]/text()')[0]
print('第一个价格：', first_price)

first_title = tree.xpath('//article[1]/h3/a/@title')   # 第 1 本书（XPath 从 1 数起）
print('第一本书：', first_title)

第一个价格： ¥51.77
第一本书： ['A Light in the Attic']


## 9.4 与 CSS 选择器互译（对照表）

| 想找什么 | CSS 选择器 | XPath |
|---|---|---|
| 带某 class 的节点 | `.product_pod` | `//*[contains(@class,'product_pod')]` |
| 某个 id | `#item-list` | `//*[@id='item-list']` |
| a 的 title 属性 | `a[title]` | `//a/@title` |
| 第一个书盒 | `.product_pod`[0] | `(//article)[1]` |
| 节点文字 | `.get_text()` | `/text()` |

In [5]:
# 用 XPath 抠 quotes.html：名言文字 + 作者
qtree = lxml_html.fromstring(quotes_html)
texts = qtree.xpath('//div[contains(@class,"quote")]//span/text()')
authors = qtree.xpath('//div[contains(@class,"quote")]//small/text()')
for t, a in zip(texts, authors):
    print(a, '|', t)

—— 温斯顿·格鲁姆 | “生活就像一盒巧克力，你永远不知道下一颗会是什么味道。”
—— 孔子 | “知之者不如好之者，好之者不如乐之者。”
—— Steve Jobs | “Stay hungry, stay foolish.”


## 9.5 随堂小练习（自己动手，空格里写）

In [6]:
# TODO：用 XPath 取出 books_index 里 nav 下那个 a 的 @href

In [7]:
# TODO：用 XPath 取 quotes.html 里每个 div.quote 的第一个 a.tag 的文字

## 来源与延伸

- XPath 入门（MDN 中文）：https://developer.mozilla.org/zh-CN/docs/Web/XPath
- lxml 官方文档：https://lxml.de/
- 下一课：把这几课抠下来的数据，真正存成 CSV 和 SQLite。